# Notebook 08 — Scientific Computing Workflow

**Module:** 00 — Orientation  
**Notebook:** 08 of 13  
**Prerequisites:** Notebooks 01–07  
**Time estimate:** 75 minutes

This notebook covers the question that every experienced computational scientist has learned the hard way: when do you use a script, a notebook, or a package? It also introduces the Wilson et al. (2017) checklist as a concrete code quality standard.

---
## Step 1 — Motivation

The most common failure mode in computational science is not wrong code — it is unstructured code. A 500-line notebook that could have been a 20-line script with a 5-function library. A one-off analysis that gets copy-pasted six times. A function that "works" but cannot be tested because it reaches into global state.

This notebook answers:
- **Script vs. notebook vs. package:** which tool for which job?
- **Wilson et al. (2017):** the 24-practice checklist that defines "good enough" for this program.
- **The one non-negotiable rule:** no code ships anywhere in this program that you cannot explain line by line, unprompted.

---
## Step 2 — Intuition

| Medium | Best for | Signs you need a different medium |
|--------|----------|-----------------------------------|
| **Notebook** | Exploration, narrative explanation, figures, teaching | You're copy-pasting cells; the notebook is >300 lines; someone else needs to run it unattended |
| **Script** | Reproducible, parameterized runs; data fetching; batch processing | You need to share the logic; it runs in CI; it gets called from other code |
| **Package** | Reusable functions shared across notebooks and modules | You've copy-pasted the same function into 3 notebooks |

`compbio_utils` is a package because the same `gc_content()` function appears in Modules 01, 05, 08, and 09. A package has tests; a notebook cell does not.

---
## Step 3 — Biological Background

*Not applicable to this workflow notebook.*

**Meta-note:** Good scientific software practices are themselves part of biology's reproducibility solution. Markowetz (2015), *"Five selfish reasons to work reproducibly"*, is a short, useful read on why reproducibility benefits the person doing the work, not just future readers.

---
## Step 4 — Mathematical Explanation

*Not applicable.*

---
## Step 5 — Computational Explanation

**Wilson et al. (2017) — the 24 practices (grouped):**

| Group | Practices |
|-------|-----------|
| **Data management** | Save raw data; create analysis-friendly data; record data processing steps; submit data to a repository |
| **Software** | Place code in a subdirectory; add a README; version control everything; use a license; cite software |
| **Collaboration** | Create an overview; explain how to contribute; make the license explicit |
| **Project organization** | Put project in a shared public repository; make requirements explicit; keep raw and processed data separate |
| **Keeping track** | Record all operations; test regularly; use assertions; submit to a code-review process |
| **Writing** | Write for humans; modularize; don't repeat yourself; plan for mistakes |

The full paper (DOI: 10.1371/journal.pcbi.1005510) is assigned in Notebook 10. This overview is enough to apply the checklist here.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Example of BAD scientific code vs. GOOD scientific code

# BAD: reaches into global state, no docstring, magic numbers, untestable
data = [0.45, 0.52, 0.61, 0.48, 0.55]  # GC contents of some sequences

def process(x):           # What does 'process' mean?
    r = []
    for i in x:
        if i > 0.5:       # Magic number — why 0.5?
            r.append(i)
    return sum(r)/len(r)  # What if r is empty?

result = process(data)
print(f"Result: {result}")

In [ ]:
# GOOD: pure function, clear name, documented, edge case handled
from typing import Sequence

GC_HIGH_THRESHOLD = 0.50  # Named constant — change in one place

def mean_gc_of_high_content_sequences(
    gc_values: Sequence[float],
    threshold: float = GC_HIGH_THRESHOLD,
) -> float:
    """Return the mean GC content of sequences above a threshold.

    Returns 0.0 if no sequences exceed the threshold.
    """
    high = [v for v in gc_values if v > threshold]
    if not high:
        return 0.0
    return sum(high) / len(high)


gc_values = [0.45, 0.52, 0.61, 0.48, 0.55]
result = mean_gc_of_high_content_sequences(gc_values)
print(f"Mean GC of high-content sequences: {result:.4f}")

# Edge case check
empty_result = mean_gc_of_high_content_sequences([], threshold=0.5)
print(f"Edge case (empty list): {empty_result}")

In [ ]:
# Cell 6.2 — Apply Wilson et al. checklist to a toy compbio_utils function
# Rate each practice: ✓ (done), ○ (partially), ✗ (not done)

checklist = [
    ("Version controlled",              "✓", "The repo is in Git"),
    ("Clear function name",              "✓", "mean_gc_of_high_content_sequences"),
    ("Docstring",                        "✓", "One-line description + return behavior"),
    ("Type hints",                       "✓", "Sequence[float] → float"),
    ("Magic numbers removed",            "✓", "GC_HIGH_THRESHOLD named constant"),
    ("Edge case handled",                "✓", "Empty list returns 0.0"),
    ("Tests written",                    "✗", "No pytest test for this function yet"),
    ("No global state",                  "✓", "Pure function — only uses its arguments"),
    ("Readable without comments",        "✓", "Variable names explain themselves"),
    ("Module-level (not notebook-only)", "○", "Still in notebook — move to compbio_utils.sequence"),
]

print(f"Wilson et al. (2017) audit — mean_gc_of_high_content_sequences")
print("=" * 65)
for practice, status, note in checklist:
    print(f"  {status}  {practice:<40}  {note}")

passes = sum(1 for _, s, _ in checklist if s == "✓")
print(f"\n{passes}/{len(checklist)} practices met. Fix the ✗ items before calling this done.")

In [ ]:
# Cell 6.3 — The 'explain every line unprompted' test
# Read this function. Without looking it up, can you explain every line?
# If not, that line is not understood yet — look it up before continuing.

def normalize_counts(counts, pseudocount: float = 0.5):
    """Return counts as log2 CPM (counts per million) with a pseudocount."""
    import math
    total = sum(counts) + pseudocount * len(counts)
    return [math.log2((c + pseudocount) / total * 1e6) for c in counts]

raw = [150, 3200, 87, 4400, 220]
normalized = normalize_counts(raw)

print("Raw counts → log2 CPM:")
for raw_val, norm_val in zip(raw, normalized):
    print(f"  {raw_val:6d}  →  {norm_val:7.3f}")

# Can you explain:
# 1. Why pseudocount is added?
# 2. What 1e6 represents?
# 3. What log2 does to the scale?
# Answer these before moving on. If you can't, look them up.

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Decision tree for script vs. notebook vs. package
decision_tree = """
  Is this analysis explorative / narrative?
      YES → Notebook (.ipynb)
      NO  ↓
  Will it be run repeatedly / by CI?
      YES → Script in scripts/
      NO  ↓
  Is the same function used in ≥ 2 places?
      YES → Add to compbio_utils (package)
      NO  → Notebook or script depending on interactivity

  Rule of thumb:
  - If you're exploring → notebook
  - If you're automating → script
  - If you're sharing reusable logic → package
  - When in doubt: notebook first, refactor to script/package when you copy-paste
"""
print(decision_tree)

---
## Step 8 — Exercises

**Exercise 1 — Wilson et al. audit:**  
Take any piece of code from a personal project (a script, a notebook cell, anything). Apply Cell 6.2's audit format to it. How many of the 10 practices listed there does it meet? Be honest. Document every failure in `exercises/08_workflow_exercises.md`.

**Exercise 2 — Explain every line:**  
Open Cell 6.3's `normalize_counts` function. Write a line-by-line explanation in plain English in `exercises/08_workflow_exercises.md`. Answer the three questions in the cell comments before consulting any reference material.

**Exercise 3 — Refactor:**  
Rewrite the BAD `process` function from Cell 6.1 as a well-named, pure, documented function that handles the empty-list edge case. Do not look at Cell 6.1's GOOD version until you've tried your own version first.

---
## Quiz — Active Recall

1. What is the difference between a pure function and a function with side effects?
2. Name three signs that a notebook cell should become a function in `compbio_utils`.
3. What does Wilson et al. (2017) mean by "make requirements explicit"?
4. Why is a named constant (`GC_HIGH_THRESHOLD = 0.5`) better than an inline magic number (`if gc > 0.5`)?
5. The rule is: no code ships that you cannot explain line by line, unprompted. What specifically does "unprompted" mean here?

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[What was your audit score in Exercise 1? What failures did you find that surprised you? Can you explain every line of the normalize_counts function from memory?]*

---
## References

- Wilson et al. (2017). DOI: 10.1371/journal.pcbi.1005510 — full paper assigned in Notebook 10
- Markowetz (2015), *Five selfish reasons to work reproducibly*, Genome Biology. DOI: 10.1186/s13059-015-0850-7

---
*Next notebook: `08_literature_review_workflow.ipynb`*